In [6]:
from catboost import CatBoostRegressor
import numpy as np
import optuna
import pandas as pd

In [7]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_train = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv", parse_dates=["visit_date"])
df_test = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv", parse_dates=["visit_date"])

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train shape: (252108, 28)
Test shape: (32019, 28)


In [8]:
from restaurant_visitor_eda.features import binary_features, categorical_features, numeric_features

features = categorical_features + numeric_features + binary_features

X_full = df_train[features]
y_full = np.log1p(df_train["visitors"].values)

In [9]:
from restaurant_visitor_eda.features import get_custom_cv_splits

cv_splits = get_custom_cv_splits(df_train, n_splits=3, val_days=39)

Fold 1: Train ends 2017-03-14| Val: 2017-03-15 to 2017-04-22
Fold 2: Train ends 2017-02-03| Val: 2017-02-04 to 2017-03-14
Fold 3: Train ends 2016-12-26| Val: 2016-12-27 to 2017-02-03


In [10]:
from optuna_integration.catboost import CatBoostPruningCallback


def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 5, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
    }

    pruning_callback = CatBoostPruningCallback(trial, "RMSE")
    cv_scores = []
    fold_iters = []

    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_train, y_train = X_full.iloc[train_idx], y_full[train_idx]
        X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]

        model = CatBoostRegressor(**params, cat_features=categorical_features)

        if fold == 0:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), callbacks=[pruning_callback])
            pruning_callback.check_pruned()
        else:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)

        best_score = model.get_best_score()["validation"]["RMSE"]
        best_iter = model.get_best_iteration()

        cv_scores.append(best_score)
        fold_iters.append(best_iter)

    trial.set_user_attr("mean_best_iter", int(np.mean(fold_iters)))

    return np.mean(cv_scores)

In [11]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow_tracking.db")
mlflow.set_experiment("CatBoost_Optuna_Tuning")

with mlflow.start_run(run_name="optuna_search_with_pruning"):
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=50)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    study.optimize(objective, n_trials=30, show_progress_bar=True)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)

    print(f"\n Best RMSE: {study.best_value:.4f}")

[I 2026-06-19 13:49:10,889] A new study created in memory with name: no-name-21e63f2b-a352-4848-a7f8-6e514b43aae0


  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_7559/1303568254.py:20: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "RMSE")


[I 2026-06-19 13:52:16,276] Trial 0 finished with value: 0.5098148195289703 and parameters: {'learning_rate': 0.015559644845853246, 'depth': 7, 'l2_leaf_reg': 8.275888684175445, 'random_strength': 4.98974942514516, 'bagging_temperature': 0.23427713116334847}. Best is trial 0 with value: 0.5098148195289703.
[I 2026-06-19 13:54:35,438] Trial 1 finished with value: 0.5071879048715032 and parameters: {'learning_rate': 0.09947767617873185, 'depth': 9, 'l2_leaf_reg': 6.116077724664203, 'random_strength': 6.2154136177039865, 'bagging_temperature': 0.10126387556476579}. Best is trial 1 with value: 0.5071879048715032.
[I 2026-06-19 13:56:25,862] Trial 2 finished with value: 0.5074822076534944 and parameters: {'learning_rate': 0.08253993753476498, 'depth': 5, 'l2_leaf_reg': 14.905468605490348, 'random_strength': 3.6694821065310395, 'bagging_temperature': 0.32902888483586124}. Best is trial 1 with value: 0.5071879048715032.
[I 2026-06-19 13:57:35,234] Trial 3 finished with value: 0.50714529783206

In [12]:
print("\n--- BEST PARAMS ---")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"\n best rmsle during CV: {study.best_value:.4f}")

optimal_iterations = study.best_trial.user_attrs["mean_best_iter"]
print(f"Optimal Iter: {optimal_iterations}")


--- BEST PARAMS ---
learning_rate: 0.09255159173066993
depth: 10
l2_leaf_reg: 18.209077875688994
random_strength: 0.21581131668641185
bagging_temperature: 0.8907843359059793

 best rmsle during CV: 0.5053
Optimal Iter: 99


In [13]:
final_params = study.best_params.copy()
final_params["iterations"] = optimal_iterations
final_params["loss_function"] = "RMSE"
final_params["eval_metric"] = "RMSE"
final_params["random_seed"] = 42

final_model = CatBoostRegressor(**final_params, cat_features=categorical_features)

final_model.fit(X_full, y_full, verbose=100)

0:	learn: 0.7681797	total: 53.4ms	remaining: 5.23s
98:	learn: 0.5069953	total: 5.11s	remaining: 0us


CatBoostRegressor(bagging_temperature=0.8907843359059793, cat_features=['air_store_id', 'air_genre_name', 'day_of_week', 'month', 'day_pattern', 'prefecture', 'district', 'block'], depth=10, eval_metric='RMSE', iterations=99, l2_leaf_reg=18.209077875688994, learning_rate=0.09255159173066993, loss_function='RMSE', random_seed=42, random_strength=0.21581131668641185)

In [14]:
X_test = df_test[features]

preds_log = final_model.predict(X_test)

preds_real_clipped = np.clip(np.expm1(preds_log), 1.0, None)

submission = pd.DataFrame(
    {
        "id": df_test["air_store_id"] + "_" + df_test["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real_clipped,
    }
)

submission_path = "submission_catboost_optuna.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,3.325069
1,air_08cb3c4ee6cd6a22_2017-04-23,13.076095
2,air_f8233ad00755c35c_2017-04-23,7.229076
3,air_234d3dbf7f3d5a50_2017-04-23,7.382226
4,air_a563896da3777078_2017-04-23,27.213715
